# LOGIVISION — Train YOLOv8n on Colab / Kaggle (free GPU)

Companion guide: [`docs/mlops/training-on-colab.md`](https://github.com/Ayalem/logivision_v2/blob/develop/docs/mlops/training-on-colab.md).

Before running, set these Colab Secrets (🔑 icon in the left rail):

| Name | Value |
|---|---|
| `MLFLOW_TRACKING_URI` | `https://…trycloudflare.com` (your tunnel) |
| `MINIO_ENDPOINT` | tunnel for MinIO :9000 |
| `MINIO_ACCESS_KEY` | from your local `.env` |
| `MINIO_SECRET_KEY` | from your local `.env` |

**Runtime → Change runtime type → T4 GPU** before running cell 1.

## 1. Setup — uv + project deps

In [ ]:
%%capture
!pip install -q uv
!git clone --branch develop https://github.com/Ayalem/logivision_v2.git
%cd logivision_v2
!uv sync --all-groups

## 2. Confirm GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))

## 3. Auth — pull secrets into env vars

On Kaggle, swap `google.colab` for `kaggle_secrets` (see the guide).

In [ ]:
import os
try:
    from google.colab import userdata
    get = userdata.get
except ImportError:  # Kaggle
    from kaggle_secrets import UserSecretsClient
    get = UserSecretsClient().get_secret

os.environ['MLFLOW_TRACKING_URI']   = get('MLFLOW_TRACKING_URI')
os.environ['MLFLOW_S3_ENDPOINT_URL'] = get('MINIO_ENDPOINT')
os.environ['AWS_ACCESS_KEY_ID']     = get('MINIO_ACCESS_KEY')
os.environ['AWS_SECRET_ACCESS_KEY'] = get('MINIO_SECRET_KEY')
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
print('MLflow ->', os.environ['MLFLOW_TRACKING_URI'])
print('MinIO  ->', os.environ['MLFLOW_S3_ENDPOINT_URL'])

## 4. DVC pull — fetch the current dataset snapshot

Writes `.dvc/config.local` so DVC speaks to MinIO with the right credentials.

In [ ]:
!uv run dvc remote modify --local minio access_key_id "$AWS_ACCESS_KEY_ID"
!uv run dvc remote modify --local minio secret_access_key "$AWS_SECRET_ACCESS_KEY"
!uv run dvc remote modify --local minio endpointurl "$MLFLOW_S3_ENDPOINT_URL"
!uv run dvc pull -v

## 5. Train

`--device cuda` overrides the `cpu` default in `ml/configs/yolov8n.yaml`.

In [ ]:
!uv run python -m ml.scripts.train --config ml/configs/yolov8n.yaml --device cuda

## 6. Push the new weights back to MinIO via DVC

The training output is under `ml/runs/<run-id>/weights/`. Track it with DVC if you want to checkpoint a specific run on top of what MLflow already stored.

In [ ]:
!uv run dvc push

## 7. Resume from a checkpoint (optional)

If the Colab runtime disconnected mid-training, mount Google Drive (or re-`dvc pull`), then resume by pointing Ultralytics at the run's `last.pt`:

```bash
!uv run python -m ml.scripts.train \
    --config ml/configs/yolov8n.yaml \
    --device cuda \
    --resume /content/drive/MyDrive/logivision/runs/<run-id>/weights/last.pt
```

(Note: the `--resume` flag is currently expected to be added in a follow-up — Ultralytics supports it natively.)